Compare the completed single pass against frozen human decisions. No API calls.

In [ ]:
import json, hashlib
from pathlib import Path
from collections import Counter
def metrics(rows, reviewer):
    def count(subset):
        return dict(Counter(r[reviewer+'_decision'] for r in subset))
    kept = [r for r in rows if r['new_keep'] is not False]
    dropped = [r for r in rows if r['new_keep'] is False]
    before, after, removed = count(rows), count(kept), count(dropped)
    n, k = len(rows), len(kept)
    def precision(c, denom, uncertain):
        return (c.get('include',0)+uncertain*c.get('unclear',0))/denom if denom else None
    before_bounds = [precision(before,n,u) for u in [0,1]]
    after_bounds = [precision(after,k,u) for u in [0,1]]
    changes = [100*(a-b) for a,b in zip(after_bounds,before_bounds)] if k and n else []
    valid = before.get('include',0)
    fp = before.get('exclude',0)
    resolved_kept = after.get('include',0)+after.get('exclude',0)
    return {'n':n,'kept_n':k,'dropped_n':len(dropped), 'before':before,'after':after,
            'dropped':removed,'before_precision_bounds':before_bounds,
            'after_precision_bounds':after_bounds,
            'precision_change_pp_bounds':[min(changes),max(changes)] if changes else [None,None],
            'resolved_before_precision':valid/(valid+fp) if valid+fp else None,
            'resolved_after_precision':after.get('include',0)/resolved_kept if resolved_kept else None,
            'human_valid_retained_n':after.get('include',0),
            'human_valid_lost_n':removed.get('include',0),
            'human_valid_retention':after.get('include',0)/valid if valid else None,
            'human_invalid_removed_n':removed.get('exclude',0),
            'human_invalid_removal_fraction':removed.get('exclude',0)/fp if fp else None,
            'unresolved_human_retained_n':after.get('unclear',0)}
baseline = json.loads('[{"audit_id": "verified:1900:167", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1916:170", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1930:1949", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1930:2156", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1931:960", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1934:1547", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1935:1146", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1936:689", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1943:278", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1947:1428", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "unclear", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1949:490", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1950:394", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1951:432", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1958:976", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1958:994", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1962:410", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1962:1212", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1969:318", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1974:1486", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1974:2087", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1976:910", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1976:2813", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:3667", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:4423", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:5495", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:1110", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:6491", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:7224", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1981:4189", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:825", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:2773", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:3867", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:5367", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:7277", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:8918", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:2644", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:4337", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:4430", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:5198", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:5568", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:6085", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:1627", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:2186", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:4191", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1986:930", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1986:1166", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1986:1897", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:984", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:3320", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:3725", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:4647", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1988:1985", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1988:2436", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1988:2930", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:1446", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:1755", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:2660", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:1577", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:1587", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:3594", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:1855", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:2790", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:3806", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:8939", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:9585", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:10318", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:13829", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1993:6666", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1994:6122", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1996:4019", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1996:4281", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1998:7306", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1999:1799", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1999:3774", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2000:2587", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:4085", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:6615", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2002:2436", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2002:4199", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2002:4800", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2002:8487", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2003:2035", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2006:793", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2007:1034", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2007:2520", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2007:2928", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:6519", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:7128", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:10465", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:12614", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:13325", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:13498", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:14036", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:2063", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:7150", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:7211", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:11622", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2010:3688", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2010:5785", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2010:7144", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1932:993", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1934:2052", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1936:952", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1937:1375", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1940:289", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1944:267", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1952:677", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1956:312", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1956:448", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1958:1236", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1966:315", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1975:2728", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1975:4701", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1976:1388", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1978:1264", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:3498", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:3741", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:4432", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:4630", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:6723", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:7038", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:7658", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1981:1586", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:5924", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:5444", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:7467", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:983", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:1832", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:2601", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:4586", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:5159", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:8331", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:10824", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1994:1910", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1995:3081", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1995:4138", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1996:2133", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1996:2137", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1998:3436", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1999:4701", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:6942", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:10568", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:11436", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2003:2141", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2005:731", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2007:477", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:10583", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:2142", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2010:1642", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2010:2040", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1928:389", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1931:1537", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1931:2278", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1932:2062", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1933:2449", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1934:854", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1937:632", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1940:233", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1949:635", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1951:200", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1953:797", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1953:938", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1958:2076", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1959:444", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1966:831", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1969:878", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1970:1066", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1970:1338", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1971:218", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1971:344", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1972:335", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1973:902", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1974:1472", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1975:3064", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1975:3851", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1976:327", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1977:1485", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:1652", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:4872", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1979:6330", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:1298", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:3271", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:3825", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1980:4291", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1981:2999", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:2221", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:2596", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:7181", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1982:8870", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:652", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:3530", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:5142", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1983:6152", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1984:2337", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1984:4814", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:1827", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:3415", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1985:4285", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1986:1381", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:1833", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1987:3769", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1988:2882", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1988:4745", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:2543", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:3049", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:3128", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:3506", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1989:4765", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:1242", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:3650", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1990:5304", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:2065", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:3979", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:4440", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:7363", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:8690", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:11419", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1991:12942", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:3087", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:5966", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:8713", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:9708", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1992:11287", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1993:1003", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1994:3634", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1994:4686", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1995:1476", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1995:3168", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1996:1505", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1997:1128", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1997:2404", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1997:3324", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1998:4096", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1999:1251", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:1999:2403", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2000:3682", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2000:3894", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:2384", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:7001", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:7684", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:7787", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2001:13488", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2002:3971", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2003:878", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2003:2215", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2004:2302", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2005:640", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2008:5461", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:11258", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2009:14639", "sampling_frame": "historical_1900_2010", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2011:25679", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2011:27875", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2011:29135", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2011:29356", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2012:3775", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2012:4197", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2012:6775", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2013:3694", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2013:4380", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2013:4554", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2014:1779", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2014:3498", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2014:4148", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2015:1017", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2015:2182", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2015:2809", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2016:2777", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2016:3328", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2016:4438", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2017:1008", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2017:1145", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2017:2752", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2018:1155", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2018:2710", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2018:3103", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2019:1847", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2019:2391", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2019:3795", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2020:7924", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2020:8899", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2020:9275", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2021:1500", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2021:2605", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2021:3627", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2022:887", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2022:7735", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2022:11034", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2023:4672", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2023:5550", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2023:8507", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2024:854", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2024:2018", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2024:2023", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2025:2458", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2025:2890", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "exclude", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2025:3376", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2026:581", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "exclude", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2026:837", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2026:1111", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}, {"audit_id": "verified:2026:1174", "sampling_frame": "modern_2011_2026", "reviewer1_decision": "include", "reviewer2_decision": "include", "saved_verifier_decision": "keep"}]')
folder = Path('/home/ec2-user/SageMaker/election_arm/gpt4omini_reverify300_20261022')
export_path = folder/'categorical_export.json'
export = json.loads(export_path.read_text())
assert export['attempted_n'] == 300
assert [r['audit_id'] for r in baseline] == [r['audit_id'] for r in export['records']]
assert all(r['saved_verifier_decision']=='keep' for r in baseline)
rows = [dict(old,new_keep=new['keep']) for old,new in zip(baseline,export['records'])]
result = {'export_sha256':hashlib.sha256(export_path.read_bytes()).hexdigest(), 'counts':export['counts'], 'errors':dict(Counter(r.get('error_type') for r in export['records'] if r['status']!='judged')), 'models':dict(Counter(r.get('response_model') for r in export['records'])), 'groups':{}}
for frame in ['historical_1900_2010','modern_2011_2026','combined_descriptive_only']:
    subset = [r for r in rows if frame=='combined_descriptive_only' or r['sampling_frame']==frame]
    result['groups'][frame] = {who:metrics(subset,who) for who in ['reviewer1','reviewer2']}
    for who,m in result['groups'][frame].items():
        print(frame, who, 'n/kept=',m['n'],m['kept_n'], 'before=',m['before'], 'after=',m['after'], 'removed=',m['dropped'])
print({k:v for k,v in result.items() if k!='groups'})
(folder/'comparison_vm.json').write_text(json.dumps(result,indent=2)+'\n')
